## 0. Setup

In [ ]:
!pip install -q torchattacks kagglehub ultralytics

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, matplotlib.pyplot as plt
from torchvision import transforms, models
from PIL import Image, ImageDraw
import glob, os, time, threading, random

SIZE    = 512        # repo-standard working resolution
EPS     = 8/255      # the AutoAttack / RobustBench standard Linf budget
SEED    = 0
SIGMA_L = 6.0        # radius of the sliding window every local statistic uses

torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)
DEVS = [f"cuda:{i}" for i in range(torch.cuda.device_count())] or ["cpu"]
NDEV = len(DEVS)
DEV  = DEVS[0]
print("devices:", DEVS)

# --- generic utilities, used by both the attack synthesis and the detector ---
def gauss1d(sigma):
    k  = int(6*sigma) | 1
    ax = torch.arange(k) - k//2
    g  = torch.exp(-(ax.float()**2)/(2*sigma**2))
    return g/g.sum()

def blur_sep(x, sigma):
    # the 2D Gaussian is an outer product, so two 1D convs are exact and ~25x cheaper
    g = gauss1d(sigma).to(x.device, x.dtype)
    k = g.numel(); p = k//2; C = x.shape[1]
    x = F.conv2d(x, g.view(1,1,1,k).repeat(C,1,1,1), padding=(0,p), groups=C)
    x = F.conv2d(x, g.view(1,1,k,1).repeat(C,1,1,1), padding=(p,0), groups=C)
    return x

gray  = lambda x: x.mean(1, keepdim=True)
local = lambda x, s=SIGMA_L: blur_sep(x, s)

## 1. Five random images from my Kaggle dataset

Two ways in, tried in order:

1. **`+ Add Input` in the Kaggle sidebar** → the dataset appears under `/kaggle/input/...`. Fastest,
   no download, works with Internet Off. **This is the one that works under Save & Run All.**
2. **`kagglehub.dataset_download(...)`** → downloads it. Needs Internet On, and **only works in an
   interactive session**: Kaggle refuses to attach a new datasource to a batch run
   (`New Datasets cannot be attached in non-interactive sessions`). So this fallback can pass in
   the editor and then fail on Save & Run All — which is exactly what happened on the second run
   of this notebook. The cell now says so instead of surfacing a backend traceback.

A note on the snippet Kaggle shows for this dataset: `KaggleDatasetAdapter.PANDAS` loads one
**tabular file** into a DataFrame. This is an image dataset, so what we want is the files on disk —
`dataset_download` returns the folder and we glob the images out of it. If the dataset also ships a
labels CSV, the PANDAS adapter is the right tool for *that file*, not for the pictures.

In [ ]:
DATASET  = "abtinzandi/obstacle-detection-dataset"
N_IMAGES = 5
EXT      = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

HOWTO = f"""
Could not reach the dataset.

  Fix (10 seconds): in the right-hand sidebar click  + Add Input , search for
  '{DATASET}', click Add. Then re-run.

Why the download fallback cannot rescue this: 'Save & Run All' is a NON-INTERACTIVE
session, and Kaggle refuses to attach a new datasource to one -- kagglehub raises
'New Datasets cannot be attached in non-interactive sessions'. The dataset has to be
attached to the notebook BEFORE the batch run starts. It works interactively, which is
why this can pass in the editor and then fail on Save & Run All.

  From the CLI instead:  kaggle kernels push -p 08-AutoAttack-Localize
  (kernel-metadata.json lists the dataset in dataset_sources, so this attaches it.)
"""

def find_images():
    for r in sorted(glob.glob("/kaggle/input/*")):            # 1. mounted via + Add Input
        f = [p for p in glob.glob(os.path.join(r, "**", "*"), recursive=True)
             if p.lower().endswith(EXT)]
        if f:
            print(f"using mounted input: {r}  ({len(f)} images)")
            return f
    try:                                                      # 2. download (interactive only)
        import kagglehub
        p = kagglehub.dataset_download(DATASET)
        f = [q for q in glob.glob(os.path.join(p, "**", "*"), recursive=True)
             if q.lower().endswith(EXT)]
        print(f"downloaded to: {p}  ({len(f)} images)")
        return f
    except Exception as e:
        print(f"kagglehub fallback failed: {type(e).__name__}: {e}")
        return []

files = find_images()
assert files, HOWTO

rng    = np.random.default_rng(SEED)
picked = [files[i] for i in rng.choice(len(files), size=min(N_IMAGES, len(files)), replace=False)]

tt  = transforms.Compose([transforms.Resize((SIZE, SIZE)), transforms.ToTensor()])
raw = torch.cat([tt(Image.open(p).convert("RGB")).unsqueeze(0) for p in picked]).to(DEV)
N   = raw.shape[0]
print(f"\n{N} random images @ {SIZE}x{SIZE}:")
for p in picked: print("  ", os.path.basename(p))

fig, ax = plt.subplots(1, N, figsize=(3.2*N, 3.4))
for i, a in enumerate(np.atleast_1d(ax)):
    a.imshow(raw[i].permute(1,2,0).cpu()); a.set_title(f"image {i}"); a.axis("off")
plt.suptitle("the 5 randomly drawn images"); plt.tight_layout(); plt.show()

## 2. Random attack regions — deliberately not rectangles

One random region per image, drawn from blob / ring / scribble / wedge. Nothing is tile-aligned,
because the detector below has no tiles: every statistic is per-pixel, so the shape of the attacked
area is unconstrained.

In [ ]:
def shape_mask(kind, seed=0, size=SIZE):
    g  = np.random.default_rng(seed)
    im = Image.new("L", (size, size), 0); d = ImageDraw.Draw(im)
    if kind == "blob":
        cx, cy = g.uniform(0.3, 0.7, 2)*size
        a = np.sort(g.uniform(0, 2*np.pi, 9)); r = g.uniform(0.14, 0.32, 9)*size
        d.polygon([(float(cx+ri*np.cos(ai)), float(cy+ri*np.sin(ai))) for ai, ri in zip(a, r)], fill=1)
    elif kind == "ring":
        c = g.uniform(0.35, 0.65, 2)*size; R = g.uniform(0.22, 0.34)*size
        d.ellipse([c[0]-R, c[1]-R, c[0]+R, c[1]+R], fill=1)
        d.ellipse([c[0]-R/2, c[1]-R/2, c[0]+R/2, c[1]+R/2], fill=0)
    elif kind == "scribble":
        y0  = g.uniform(0.25, 0.75)*size
        pts = [(0.12*size, y0)] + [(0.12*size + i*0.12*size,
                y0 + 0.18*size*np.sin(i*g.uniform(0.9, 1.6))) for i in range(1, 7)]
        d.line([(float(a), float(b)) for a, b in pts], fill=1, width=int(0.05*size), joint="curve")
    else:                                                     # wedge
        c = g.uniform(0.3, 0.7, 2)*size; R = g.uniform(0.3, 0.45)*size; s = g.uniform(0, 360)
        d.pieslice([c[0]-R, c[1]-R, c[0]+R, c[1]+R], s, s + g.uniform(70, 140), fill=1)
    return torch.tensor(np.array(im), dtype=torch.float32).view(1, 1, size, size)

KINDS = ["blob", "ring", "scribble", "wedge"]
masks = torch.cat([shape_mask(KINDS[i % len(KINDS)], seed=100+i) for i in range(N)]).to(DEV)
print("region coverage per image (% of pixels):", [f"{m.mean().item()*100:.1f}" for m in masks])

fig, ax = plt.subplots(1, N, figsize=(3.2*N, 3.4))
for i, a in enumerate(np.atleast_1d(ax)):
    a.imshow(masks[i,0].cpu(), cmap="gray"); a.set_title(KINDS[i % len(KINDS)]); a.axis("off")
plt.suptitle("attack regions (ground truth) - arbitrary shapes"); plt.tight_layout(); plt.show()

## 3. The victim model and real AutoAttack perturbations

AutoAttack needs gradients through a classifier, so the perturbations are crafted against a
pretrained **ResNet-50**, with labels taken from the model's own clean predictions (the standard
convention for unlabelled data). §10 then measures what that same perturbation does to a **YOLO
obstacle detector** — the task this dataset is actually for.

The four AutoAttack components are run **separately** rather than as the ensemble, because "which
component does the detector catch?" is the informative question; the ensemble would only report
their union. Each is a real `torchattacks` implementation.

**One honest caveat.** AutoAttack optimizes over the whole image and has no masked variant, so δ is
computed on the full image and then **confined to the region afterwards** (`x + m·δ`). The noise
*statistics* the detector sees are genuine AutoAttack output, which is what is under test, but the
confined perturbation is no longer optimal for that region — so the printed fooling rate is a lower
bound. A hand-rolled **masked PGD**, which applies the mask at every step, is included as the
properly-confined reference.

Both T4s are used: the batch is split in half and each half attacked on its own GPU in a thread.
The attack is iterative and slow, so this is where parallelism actually pays — unlike the detector,
which `07` showed needs nothing beyond batching.

In [ ]:
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1,3,1,1)
STD  = torch.tensor([0.229, 0.224, 0.225]).view(1,3,1,1)

class Wrapped(nn.Module):
    # torchattacks expects a model that takes [0,1] images, so fold normalization inside
    def __init__(self, net, dev):
        super().__init__()
        self.net = net.eval().to(dev)
        self.register_buffer("m", MEAN.to(dev)); self.register_buffer("s", STD.to(dev))
    def forward(self, x):
        return self.net((x - self.m)/self.s)

_w   = models.ResNet50_Weights.IMAGENET1K_V2
nets = {d: Wrapped(models.resnet50(weights=_w), d) for d in DEVS}
for n in nets.values():
    for p in n.parameters(): p.requires_grad_(False)

with torch.no_grad():
    labels = nets[DEV](raw).argmax(1)
print("clean ResNet-50 predictions (used as attack labels):", labels.tolist())

In [ ]:
import torchattacks

def build(name, model):
    # torchattacks kwargs shift between versions, so an unavailable attack is skipped
    # rather than killing the run
    try:
        if name == "APGD-CE":
            return torchattacks.APGD(model, norm="Linf", eps=EPS, steps=50, n_restarts=1, loss="ce")
        if name == "APGD-T":
            return torchattacks.APGDT(model, norm="Linf", eps=EPS, steps=50,
                                      n_restarts=1, n_classes=1000)
        if name == "FAB-T":
            return torchattacks.FAB(model, norm="Linf", eps=EPS, steps=50,
                                    n_restarts=1, n_classes=1000, multi_targeted=True)
        if name == "Square":
            return torchattacks.Square(model, norm="Linf", eps=EPS, n_queries=2000, n_restarts=1)
        if name == "AutoAttack":
            return torchattacks.AutoAttack(model, norm="Linf", eps=EPS,
                                           version="standard", n_classes=1000)
    except Exception as e:
        print(f"  [skip] {name}: {type(e).__name__}: {e}")
    return None

AA_NAMES = ["APGD-CE", "APGD-T", "FAB-T", "Square"]   # append "AutoAttack" for the full ensemble

def run_split(name):
    # attack the batch across all available GPUs, one thread per device
    chunks = np.array_split(np.arange(N), NDEV)
    out    = [None]*NDEV
    def work(gi):
        idx = chunks[gi]
        if len(idx) == 0: return
        d = DEVS[gi]
        if d.startswith("cuda"): torch.cuda.set_device(gi)
        atk = build(name, nets[d])
        if atk is None: return
        # .detach(): torchattacks returns tensors still attached to the attack graph, which
        # later makes every downstream .numpy()/imshow raise. Cost of not doing this: the
        # first run of this notebook crashed in section 9.
        out[gi] = atk(raw[idx].to(d), labels[idx].to(d)).detach().to(DEV)
    ts = [threading.Thread(target=work, args=(g,)) for g in range(NDEV)]
    for t in ts: t.start()
    for t in ts: t.join()
    got = [o for o in out if o is not None]
    if len(got) != sum(1 for c in chunks if len(c)): return None
    return torch.cat(got)

adv_full = {}
for nm in AA_NAMES:
    t0 = time.perf_counter()
    a  = run_split(nm)
    if a is not None:
        adv_full[nm] = a
        print(f"{nm:11s} done in {time.perf_counter()-t0:6.1f}s   "
              f"max|delta| = {(a-raw).abs().max().item()*255:.2f}/255")
assert adv_full, "no attack ran - check the torchattacks version"

In [ ]:
def rms_in(v, m):
    mm = m.expand_as(v) > 0.5
    return (v[mm]**2).mean().sqrt()

def masked_pgd(x, y, m, steps=50, alpha=EPS/8):
    # properly-confined reference: the mask is applied at EVERY step, so the optimizer
    # only ever spends budget inside the region
    net = nets[DEV]
    d   = (torch.rand_like(x)*2 - 1)*EPS*m
    for _ in range(steps):
        d.requires_grad_(True)
        g, = torch.autograd.grad(F.cross_entropy(net((x+d).clamp(0,1)), y), d)
        d  = (d.detach() + alpha*g.sign()*m).clamp(-EPS, EPS)
    return (x + d*m).clamp(0, 1)

def synth(kind, x, m, ref_rms, seed=0):
    # controls AutoAttack cannot produce, RMS-matched inside the mask to the real attack so
    # that nothing wins on raw energy alone (a mistake 07 paid for)
    g = torch.Generator().manual_seed(seed)
    n = torch.randn(x.shape, generator=g).to(x.device)
    d = blur_sep(n, 1.5) if kind == "L2-smooth" else blur_sep(n, 8.0)
    return (x + d/(rms_in(d, m) + 1e-10)*ref_rms*m).clamp(0, 1)

## 4. The detector: five per-pixel features, batched, no tiles

Every feature is a local statistic computed at full resolution. `07` established why there are no
tiles: `conv2d` zero-pads each tile independently, fabricating a black neighbour, which inflated
the coarse bands by 24× on interior tiles.

Six features, spanning three different ideas about what makes a perturbation unnatural:
**energy** (`07`'s: how much fine detail), **SRM / slope / -rho1** (what *shape* that fine detail
has), and **chroma** (whether the colour channels agree about it). The last one is new in v2 and
is the only one that survives attacks that are not white noise.

In [ ]:
# Kirchner-Vahid 5x5 residual. The SRM / spatial-rich-model family is the literature's strongest
# hand-crafted adversarial-detection feature set (arXiv:1806.09186).
KV = torch.tensor([[-1,  2,  -2,  2, -1],
                   [ 2, -6,   8, -6,  2],
                   [-2,  8, -12,  8, -2],
                   [ 2, -6,   8, -6,  2],
                   [-1,  2,  -2,  2, -1]], dtype=torch.float32)/12.

def _pad_to(t, ref):
    return F.pad(t, (0, ref.shape[-1]-t.shape[-1], 0, ref.shape[-2]-t.shape[-2]), mode="replicate")

def f_energy(x):
    # 07's statistic: log local energy of the finest band. Scale-DEPENDENT, hence its failure.
    return torch.log(local(gray(x - blur_sep(x, 1.0))**2) + 1e-10)

def f_srm(x):
    return torch.log(local(F.conv2d(gray(x), KV.view(1,1,5,5).to(x.device), padding=2)**2) + 1e-10)

def f_slope(x):
    # THE feature: local 2nd-difference energy over 1st-difference energy.
    # Contrast cancels in the ratio. White noise -> 3.0, natural content -> well below.
    g   = gray(x)
    d1x = g[..., :, 1:] - g[..., :, :-1]
    d1y = g[..., 1:, :] - g[..., :-1, :]
    d2x = g[..., :, 2:] - 2*g[..., :, 1:-1] + g[..., :, :-2]
    d2y = g[..., 2:, :] - 2*g[..., 1:-1, :] + g[..., :-2, :]
    e1  = local(_pad_to(d1x, g)**2 + _pad_to(d1y, g)**2)
    e2  = local(_pad_to(d2x, g)**2 + _pad_to(d2y, g)**2)
    return e2/(e1 + 1e-10)

def f_rho1(x):
    # negated lag-1 autocorrelation of the high-pass residual: also a ratio, also scale-free.
    # Natural texture is spatially correlated; injected noise is not.
    r   = gray(x - blur_sep(x, 1.0))
    num = _pad_to(local(r[..., :, 1:]*r[..., :, :-1]), r)
    return -num/(local(r**2) + 1e-10)

def f_slope_coarse(x):
    # the same ratio one octave down - the band where a low-frequency poison should live
    b  = blur_sep(gray(x), 2.0)
    d1 = b[..., :, 2:] - b[..., :, :-2]
    d2 = b[..., :, 4:] - 2*b[..., :, 2:-2] + b[..., :, :-4]
    return local(_pad_to(d2, b)**2)/(local(_pad_to(d1, b)**2) + 1e-10)

def f_chroma(x):
    # CROSS-CHANNEL DECORRELATION. Every attack here perturbs R, G and B independently, but
    # natural fine detail is luminance-dominated and strongly channel-correlated. So the
    # chroma part of the fine residual is nearly pure attack. This is the feature that
    # carries the non-white attacks (Square, smooth L2) that every residual feature misses.
    hp = x - blur_sep(x, 1.5)                              # [N,3,H,W], per channel
    return torch.log(local((hp - hp.mean(1, keepdim=True)).pow(2).mean(1, keepdim=True)) + 1e-10)

FEATS = {"energy (07)": f_energy, "SRM": f_srm, "slope": f_slope,
         "-rho1": f_rho1, "slope-coarse": f_slope_coarse, "chroma": f_chroma}

def two_sided(f):
    # |robust z| against each image's OWN median/MAD -> [N,H,W].
    # Per-image because cross-image calibration is hopeless (07 measured a 6x spread), and
    # ABSOLUTE because the deviation is signed by attack type: Linf flattens the local spectrum,
    # smooth L2 steepens it. |.| catches both; a one-sided score INVERTS on half the zoo.
    return _z(f).abs()[:, 0]

def one_sided(f):
    # 07's actual gate: upper tail only, `median + k*MAD`. Kept so the baseline in the money
    # figure is the detector 07 really ran, not a two-sided variant it never used.
    return _z(f)[:, 0]

def _z(f):
    v   = f.flatten(1)
    med = v.median(1).values.view(-1,1,1,1)
    mad = (f - med).flatten(1).abs().median(1).values.view(-1,1,1,1) + 1e-10
    return (f - med)/(1.4826*mad)

print(f"{len(FEATS)} features:", list(FEATS))

In [ ]:
with torch.no_grad():
    print(f"{'feature':14s} {'clean medians across the 5 images':46s} "
          f"{'between-image spread':>21s}   {'attack shift':>12s}")
    print("-"*100)
    for nm, fn in FEATS.items():
        v = [fn(raw[i:i+1]).median().item() for i in range(N)]
        # max/min is nonsense for a signed feature (-rho1 straddles zero and reported 0.03x
        # in the first run). Compare the between-image RANGE with the shift the attack causes,
        # in the same units - that ratio is what actually decides if one threshold can work.
        rng   = max(v) - min(v)
        a     = ZOO[REF]
        shift = abs(np.median([fn(a[i:i+1])[..., SUPPORT[REF][i]].median().item()
                               - v[i] for i in range(N)]))
        verdict = "signal > confound" if shift > rng else "confound > signal"
        print(f"{nm:14s} {' '.join(f'{q:8.3f}' for q in v):46s} "
              f"{rng:21.3f}   {shift:12.3f}  {verdict}")

print("\n"
      "This is 07's diagnosis restated as a ratio: a feature can only carry ONE threshold\n"
      "across images if the attack moves it further than clean photos differ from each other.\n"
      "Any row reading 'confound > signal' cannot be globally thresholded, whatever its AUC.")

## 5. Assembling the attack zoo

The four AutoAttack components with δ confined to each image's region, the properly-confined masked
PGD, and the two RMS-matched controls. The reference RMS is taken from APGD-CE, so every synthetic
control carries **the same perturbation energy inside the mask** as the real attack — otherwise a
louder attack wins for free, which is exactly how `07` once concluded "low-frequency is harder"
from an attack that was merely 4× weaker.

In [ ]:
REF     = "APGD-CE" if "APGD-CE" in adv_full else list(adv_full)[0]
REF_RMS = rms_in(adv_full[REF] - raw, masks)
print(f"reference RMS inside the region ({REF}): {REF_RMS.item():.4f}")

ZOO = {nm: (raw + (a - raw)*masks).clamp(0, 1) for nm, a in adv_full.items()}
ZOO["masked-PGD"] = masked_pgd(raw, labels, masks).detach()
for k in ["L2-smooth", "low-freq"]:
    ZOO[k] = torch.cat([synth(k, raw[i:i+1], masks[i:i+1], REF_RMS, seed=i) for i in range(N)])

# --- the two ground truths -------------------------------------------------------------
# REGION  = the area the attacker chose to touch.
# SUPPORT = the pixels inside it that actually carry a perturbation, |delta| > eps/4.
# These differ a LOT, because AutoAttack spends its budget where the CLASSIFIER is
# sensitive, not uniformly. The first run of this notebook scored everything against
# REGION and so charged the detector for missing pixels that carry no signal at all.
REGION  = masks[:, 0] > 0.5
SUPPORT = {nm: ((a - raw).abs().amax(1) > EPS/4) & REGION for nm, a in ZOO.items()}

with torch.no_grad():
    # clean and attacked scored in ONE forward pass: separate passes can pick different
    # cuDNN kernels and flip argmax on near-ties, which fakes a 100% fooling rate
    logits_c = nets[DEV](raw)
    top2 = logits_c.topk(2, dim=1).values
    print(f"clean top-1 margin per image: {[f'{v:.2f}' for v in (top2[:,0]-top2[:,1]).tolist()]}")
    print("(a small margin means argmax flips on almost any perturbation - read 'fooled' with that"
          " in mind)\n")
    print(f"{'attack':12s} {'RMS in region':>13s} {'max|d| /255':>12s} "
          f"{'support % of region':>20s} {'fooled':>8s}")
    print("-"*70)
    for nm, a in ZOO.items():
        both = nets[DEV](torch.cat([raw, a]))
        pc, pa = both[:N].argmax(1), both[N:].argmax(1)
        frac = (SUPPORT[nm].float().sum()/REGION.float().sum()).item()*100
        print(f"{nm:12s} {rms_in(a-raw, masks).item():13.4f} "
              f"{(a-raw).abs().max().item()*255:12.2f} {frac:19.0f}% "
              f"{f'{(pa!=pc).sum().item()}/{N}':>8s}")

print("\n"
      "The 'support' column is the important one. A saturated perturbation would fill 100% of\n"
      "the region; anything well below that means the attack left most of its own region\n"
      "untouched, so REGION is a wrong label to score a localizer against. Section 6 reports\n"
      "both and the gap between them is the size of the mistake.")

### The invariance claim, verified on these photos

`07`'s diagnosis was that clean-image energy varies far more *between* photos than an attack shifts
it *within* one. Below: how far each feature's clean median moves across these 5 real images. The
energy features are logs, so their spread is printed in nats as well as the equivalent linear
factor; the ratio features are printed as a plain max/min.

A feature can only carry a global threshold if this spread is small.

# Localizing AutoAttack noise: what survived contact with real photographs

**Read this first — it is the third version, and it is a record of two wrong predictions.**

v1 was designed on synthetic 1/f images and predicted that a contrast-invariant *spectral slope*
statistic would solve localization. It ran on real photographs and scored **AUC 0.55 — chance**.

v2 diagnosed that failure as a ground-truth error and predicted the fix was worth ~0.13 AUC. It
ran, and the fix was worth **0.000**. The prediction was wrong for a reason worth keeping (below).

What is left after both corrections is smaller than either version claimed and is measured rather
than argued: **a 6–11× improvement in IoU over `07` at a third of its false-positive budget**, one
genuinely new feature, and a low-frequency attack that still defeats everything.

### Finding 1: my ground-truth hypothesis was wrong

I predicted the biggest error was the ground truth. The reasoning: v1's own table showed APGD-CE
with an in-region RMS of 0.0201 against 0.0314 for a saturated perturbation, which I read as "only
~40% of the region carries any perturbation". Scoring against the pixels actually perturbed should
then have been worth ~0.13 AUC.

**Measured, it is worth nothing.** Section 5 reports the support at **99%** of the region, and
section 6 scores SRM both ways: **0.696 vs 0.696**. The RMS shortfall is because APGD's δ is
*graded in amplitude*, not because it is *absent* over part of the region — a distinction I could
not make from a single RMS number and should not have asserted from one.

The support/region split stays in the notebook because it is now *measured* rather than assumed,
and because it is the right instrument if an attack ever does leave its region partly untouched.
It just is not what was wrong here.

### Finding 2: v1's headline statistic did not survive real photographs

`slope` was designed and validated on synthetic 1/f images, which are spatially **stationary** —
one median and MAD describe the whole image. Real photographs are not: sky and foliage within one
frame differ more than two photographs do. The prototype had removed the exact confound the
statistic was built to survive. On real photos the fine scale is also already noise-like (clean
slope 1.16–2.59 against 3.0 for white noise), leaving little headroom.

**A synthetic validation set that lacks the confound you are trying to defeat will confirm
anything.** That is the durable lesson here.

### Finding 3: what actually works, measured on this dataset

Per-pixel AUC against the perturbation support, pooled over 5 images:

| feature | APGD-CE / APGD-T / masked-PGD | Square | smooth L2 | low-freq |
|---|---|---|---|---|
| energy (`07`'s gate, 2-sided) | 0.39–0.40 | 0.41 | 0.44 | 0.45 |
| `07` gate as actually run (1-sided) | 0.60 | 0.59 | 0.51 | 0.50 |
| slope (v1's headline) | 0.55 | 0.56 | 0.44 | 0.45 |
| **SRM** | **0.69–0.70** | 0.45 | 0.45 | 0.45 |
| **chroma decorrelation** | 0.65–0.66 | **0.70** | **0.52** | 0.41 |

And IoU at a 5% clean false-positive rate — the metric `07` reported, where it scored 0.05–0.16
**at 15%**:

| detector | APGD-CE | Square | smooth L2 | low-freq |
|---|---|---|---|---|
| `07` gate | 0.022 | 0.023 | 0.026 | 0.026 |
| SRM | **0.142** | 0.023 | 0.022 | 0.020 |
| chroma | 0.122 | **0.261** | **0.126** | 0.007 |

So the real gain over `07` is **6× on L∞ and 11× on Square at a third of the false-positive
budget** — worthwhile, and far short of a solved problem. Nothing here is accurate enough to drive
an inpainter.

**Cross-channel decorrelation is the one genuinely new thing.** Attacks perturb R, G and B
independently while natural fine detail is luminance-dominated, so the chroma part of the fine
residual is nearly pure attack. Two things follow, both visible in the output:

- It is the **only feature that catches Square Attack** (0.70 against 0.45 for SRM). Square is
  piecewise-*constant* over patches, so it adds no high-frequency residual for SRM to find.
- It is the **only feature whose attack shift exceeds its between-image spread** (3.25 vs 2.57 in
  section 5's invariance table). Every other feature reads `confound > signal`, meaning no single
  global threshold can work for it — which is `07`'s original diagnosis, now measured per feature.

**No single detector wins everywhere**, so the notebook reports SRM and chroma separately rather
than pretending a blend dominates: `mean(SRM, chroma)` is middling on both (0.135 / 0.161).

**Still unsolved: the low-frequency poison.** Nothing exceeds 0.48 on it, and chroma is actively
*worse* than chance (0.41). Same open problem `06` predicted and `07` measured.

### Where this sits in the sequence

[`../06-Attack-Benchmark/`](../06-Attack-Benchmark/) predicted the high-frequency energy gate would
fail on attacks that are not broadband. [`../07-Attack-Repair/`](../07-Attack-Repair/) measured that
failure and reported a **negative result**: IoU 0.05–0.16 at ~15% false positives, because
**median band energy varies ~6× between clean photos while a bounded attack lifts local energy only
~2.5×.** This notebook does beat that — but by fixing the measurement and finding a feature
`07` never tried, not by the route v1 predicted.

### One thing from v1 that did survive

**The score is two-sided** — `|robust z|` against each image's own median, rather than an upper
tail. L∞ noise pushes these statistics one way and smooth L2 noise pushes them the other, so a
one-sided score does not merely miss half the zoo, it ranks it backwards. `07`'s gate was
one-sided, and it is kept in that form below as the baseline it actually was.

### What this notebook measures

Real `torchattacks` AutoAttack components (APGD-CE, APGD-T, FAB-T, Square) crafted against a
ResNet-50 and confined to random non-rectangular regions on 5 random images from my
[obstacle-detection dataset](https://www.kaggle.com/datasets/abtinzandi/obstacle-detection-dataset),
plus RMS-matched smooth-L2 and low-frequency controls that AutoAttack cannot produce.

Reported under `07`'s methodology rules, which exist because breaking them produced confidently
wrong conclusions before: **per-pixel ROC-AUC** as the threshold-free primary metric, **IoU at a
false-positive rate calibrated on clean images only**, and **leave-one-image-out** validation with
the fusion **fitted on APGD-CE alone** and tested on every other attack — so no number comes from
training on its own test attack.

**Environment: Kaggle, accelerator GPU T4 ×2, Internet On.** Both T4s are used for the attack (the
expensive iterative part); the detector is a handful of convolutions and runs batched on one.
Expect roughly 10–15 minutes end to end.

## 6. Money table 1 — per-pixel ROC-AUC, every feature × every attack

AUC is threshold-free, which separates "is the signal there?" from "did I pick a good threshold?".
`07` conflated those and it cost a wrong conclusion. Scores are **pooled across all 5 images** —
one global ranking, the hard version, because a per-image AUC would hide the calibration problem
that broke the energy statistic.

Scored against the **effective support** — the pixels each attack actually perturbed. The same
table against the drawn region is printed underneath so the difference is visible; that difference
was the single largest error in v1.

Every feature is scored **two-sided**, so the comparison is like-for-like. The gate `07` actually
deployed was **one-sided** (`median + k·MAD`, upper tail only), so that exact variant is printed as
the reference baseline and is the baseline curve in §8 — comparing against a two-sided version it
never ran would be a strawman.

In [ ]:
def auc(score, label):
    # rank-based AUC, on GPU. score/label: 1-D tensors
    s = score.float().flatten(); y = label.flatten() > 0.5
    npos = int(y.sum()); nneg = y.numel() - npos
    if npos == 0 or nneg == 0: return float("nan")
    r = torch.empty_like(s)
    r[s.argsort()] = torch.arange(1, s.numel()+1, device=s.device, dtype=s.dtype)
    return ((r[y].sum() - npos*(npos+1)/2)/(npos*nneg)).item()

# An attack whose support is EMPTY has nothing to localize, so every AUC against it is nan.
# FAB-T is exactly that: a MINIMUM-NORM attack, measured in section 5 at max|delta| = 0.02/255,
# about 1/400 of the budget. Excluding it is not hiding a failure - scoring a localizer on an
# image that carries no perturbation is meaningless.
SCORED = [nm for nm in ZOO if SUPPORT[nm].sum() > 0]
EMPTY  = [nm for nm in ZOO if nm not in SCORED]
if EMPTY:
    print(f"excluded, no perturbation to find: {EMPTY}")
    print("  a minimum-norm attack landing ~1/400 of the eps budget leaves nothing to localize,")
    print("  which is a fact about the attack, not a result about the detector.")
    print()

with torch.no_grad():
    AUCS = {nm: {k: auc(two_sided(fn(ZOO[nm])).flatten(), SUPPORT[nm].flatten())
                 for k, fn in FEATS.items()} for nm in SCORED}
    AUCS_R = {nm: auc(two_sided(f_srm(ZOO[nm])).flatten(), REGION.flatten()) for nm in SCORED}

hdr = f"{'attack':12s}" + "".join(f"{k:>14s}" for k in FEATS)
print("AUC vs the EFFECTIVE PERTURBATION SUPPORT")
print(hdr); print("-"*len(hdr))
for nm in SCORED:
    print(f"{nm:12s}" + "".join(f"{AUCS[nm][k]:14.3f}" for k in FEATS))

# the baseline 07 actually ran, for reference: same energy feature, upper tail only
with torch.no_grad():
    ONE = {nm: auc(one_sided(f_energy(ZOO[nm])).flatten(), SUPPORT[nm].flatten())
           for nm in SCORED}
print(f"\n{'(reference) 07 gate, one-sided energy:':40s}"
      + "  ".join(f"{nm}:{ONE[nm]:.3f}" for nm in SCORED))

print("\nHow much the ground-truth definition alone is worth (SRM scored both ways):")
print(f"  {'attack':12s} {'vs SUPPORT':>11s} {'vs REGION':>11s}")
for nm in SCORED:
    print(f"  {nm:12s} {AUCS[nm]['SRM']:11.3f} {AUCS_R[nm]:11.3f}")

# every claim below is checked against the numbers rather than asserted
best = {nm: max(FEATS, key=lambda k: AUCS[nm][k]) for nm in SCORED}
print("\nbest feature per attack:", ", ".join(f"{nm}={best[nm]}" for nm in SCORED))
lin = [nm for nm in SCORED if nm in (REF, "APGD-T", "masked-PGD")]
print(f"\nmean AUC on the Linf attacks {lin}:")
for k in FEATS:
    print(f"  {k:14s} {np.nanmean([AUCS[nm][k] for nm in lin]):.3f}")

## 7. Money table 2 — the learned fusion, leave-one-image-out, fitted on APGD-CE only

A tiny logistic regression over the six two-sided features — 7 parameters. **This section is kept
because it fails**: fitted on one attack, it transfers worse than the plain features it is built
from. The protocol is the strict one:

- **Leave-one-image-out.** Fit on four images, score the fifth. No image is ever scored by a model
  that saw it.
- **Fitted on APGD-CE alone**, then tested on every attack, so the FAB-T, Square, L2 and
  low-frequency columns are all **transfer**, not fitted performance. This is the honest version of
  the "small learned per-tile classifier" that the [`../05-Noise-Gate/`](../05-Noise-Gate/) research
  log has listed as the next step from the start.
- Clean images join the training set as pure negatives.

Five images is a small training set — which is why the protocol is leave-one-out; that is what
makes a number from five images mean anything. Watch the "beats the best single feature" count at
the bottom: if a fitted combination cannot beat its own inputs, the right move is to ship the
unfitted score, which is what §8 and §9 use.

In [ ]:
SUB      = 4      # pixel stride when fitting (speed only)
TRAIN_ON = REF    # fit on APGD-CE; every other attack is transfer

def featmat(x):
    # [1,3,H,W] -> [P,F] matrix of two-sided features
    return torch.cat([two_sided(fn(x)).flatten().unsqueeze(1) for fn in FEATS.values()], 1)

def fit_logreg(X, y, iters=400, lr=0.5):
    mu, sd = X.mean(0), X.std(0) + 1e-8
    Xn = (X - mu)/sd
    w  = torch.zeros(X.shape[1], device=X.device, requires_grad=True)
    b  = torch.zeros(1, device=X.device, requires_grad=True)
    opt = torch.optim.Adam([w, b], lr=lr)
    for _ in range(iters):
        opt.zero_grad()
        F.binary_cross_entropy_with_logits(Xn@w + b, y).backward()
        opt.step()
    return w.detach(), b.detach(), mu, sd

def apply_logreg(m, X):
    w, b, mu, sd = m
    return ((X - mu)/sd)@w + b

with torch.no_grad():
    Fatt   = {i: featmat(ZOO[TRAIN_ON][i:i+1])[::SUB]        for i in range(N)}
    Fclean = {i: featmat(raw[i:i+1])[::SUB]                  for i in range(N)}
    ysub   = {i: SUPPORT[TRAIN_ON][i].flatten()[::SUB].float() for i in range(N)}

LOIO = []
for held in range(N):
    tr = [i for i in range(N) if i != held]
    X  = torch.cat([Fatt[i] for i in tr] + [Fclean[i] for i in tr])
    y  = torch.cat([ysub[i] for i in tr] + [torch.zeros_like(ysub[i]) for i in tr])
    LOIO.append(fit_logreg(X, y))

def fuse_score(z):
    # image i is always scored by the model that did NOT see image i
    with torch.no_grad():
        return torch.cat([apply_logreg(LOIO[i], featmat(z[i:i+1])).view(1, SIZE, SIZE)
                          for i in range(z.shape[0])])

with torch.no_grad():
    FUSED    = {nm: fuse_score(ZOO[nm]) for nm in SCORED}
    FUSE_AUC = {nm: [auc(FUSED[nm][i].flatten(), SUPPORT[nm][i].flatten()) for i in range(N)]
                for nm in SCORED}

print(f"{'attack':12s} {'fusion AUC':>11s}   {'best single feature':>28s}   per-held-out-image")
print("-"*90)
beats = 0
for nm in SCORED:
    b   = max(FEATS, key=lambda k: AUCS[nm][k])
    per = FUSE_AUC[nm]
    beats += np.nanmean(per) > AUCS[nm][b]
    tag = "   <- fitted on this" if nm == TRAIN_ON else ""
    print(f"{nm:12s} {np.nanmean(per):11.3f}   {b + f' ({AUCS[nm][b]:.3f})':>28s}   "
          + " ".join(f"{q:.2f}" for q in per) + tag)
print(f"\nthe fitted fusion beats the best single feature on {beats}/{len(SCORED)} attacks")

W  = torch.stack([m[0] for m in LOIO]).mean(0)
wd = dict(zip(FEATS, W.tolist()))
print("\nmean learned weight (positive = a larger |deviation| means attacked):")
for nm, q in wd.items(): print(f"  {nm:14s} {q:+.3f}")

# self-checking, not asserted. v1 claimed the fusion 'rediscovers' a ratio by putting
# opposite signs on the two energy features. That happened on synthetic images and is
# reported below only if it actually recurs. What matters more is whether the fitted
# model beats the plain features it was built from.
print()
if beats == 0:
    print('The fitted fusion did not beat a single one of its own inputs. Fitting on one')
    print('attack buys accuracy on that attack and loses it everywhere else, which is why')
    print('sections 8 and 9 use the UNFITTED mean(SRM, chroma) instead. A negative result')
    print('about learning, obtained under a protocol strict enough to trust.')
else:
    print(f'The fitted fusion beat the best single feature on {beats} of {len(SCORED)} attacks.')
    print(f'With only {N} images, re-check that at a larger N before relying on it.')
if wd['SRM']*wd['energy (07)'] < 0:
    print()
    print('Aside: the two energy features again took opposite signs, i.e. the model built a')
    print('ratio - a scale-invariant shape statistic - out of two scale-dependent ones.')

## 8. Money figure — IoU vs. clean false positives

`07`'s hardest-won rule: **a fixed threshold proves nothing**, because any detector can raise its
IoU by flagging more pixels. So the threshold is chosen **on clean images only**, to hit a target
false-positive rate, then applied unchanged to the attacked images. Better detectors sit **up and
to the left**.

The grey band and dashed line mark where `07` landed: IoU 0.05–0.16 at ~15% false positives.
IoU is computed against each attack's effective support, and `mean(SRM, chroma)` — the unfitted
score — is the one to watch.

In [ ]:
FPS            = [0.005, 0.01, 0.02, 0.05, 0.10, 0.20]
CURVE_ATTACKS  = [k for k in [REF, "Square", "L2-smooth", "low-freq"] if k in SCORED]

def iou_curve(score_of):
    # score_of(batch) -> [N,H,W]. Threshold comes from CLEAN images only; IoU is on attacked,
    # scored against each attack's own effective support.
    clean = score_of(raw).flatten().float()
    att   = {nm: score_of(ZOO[nm]) for nm in CURVE_ATTACKS}      # scored once, not per threshold
    out = []
    for fp in FPS:
        thr  = torch.quantile(clean[::7], 1 - fp)
        row  = []
        for nm in CURVE_ATTACKS:
            p = att[nm] > thr; g = SUPPORT[nm]
            u = (p | g).sum().item()
            row.append((p & g).sum().item()/u if u else 1.0)
        out.append(row)
    return np.array(out)                                          # [len(FPS), len(CURVE_ATTACKS)]

# The primary detector: the mean of two two-sided z-scores, no fitting at all. SRM carries the
# white L-inf attacks, chroma carries the ones that are not white (Square, smooth L2).
def srm_chroma(z):
    return (two_sided(f_srm(z)) + two_sided(f_chroma(z)))/2

with torch.no_grad():
    curves = {"07 gate (energy, 1-sided)": iou_curve(lambda z: one_sided(f_energy(z))),
              "slope (from the prototype)": iou_curve(lambda z: two_sided(f_slope(z))),
              "SRM":                       iou_curve(lambda z: two_sided(f_srm(z))),
              "chroma":                    iou_curve(lambda z: two_sided(f_chroma(z))),
              "mean(SRM, chroma)":         iou_curve(srm_chroma),
              "learned fusion (LOIO)":     iou_curve(fuse_score)}

fig, axes = plt.subplots(1, len(CURVE_ATTACKS), figsize=(4.3*len(CURVE_ATTACKS), 4.2), sharey=True)
axes = np.atleast_1d(axes)
for j, (ax, nm) in enumerate(zip(axes, CURVE_ATTACKS)):
    for dn, c in curves.items():
        ax.plot(np.array(FPS)*100, c[:, j], "o-", label=dn)
    ax.axhspan(0.05, 0.16, color="gray", alpha=.18)
    ax.axvline(15, ls="--", c="gray", lw=1)
    ax.set_xscale("log"); ax.set_xlabel("clean false positives (% of pixels)")
    ax.set_title(nm); ax.grid(alpha=.3)
axes[0].set_ylabel("localization IoU"); axes[0].legend(fontsize=8)
plt.suptitle("Better = up and to the LEFT.  Grey band / dashed line = where 07-Attack-Repair landed.")
plt.tight_layout(); plt.show()

i5 = FPS.index(0.05)
print(f"{'detector':26s}" + "".join(f"{n[:11]:>13s}" for n in CURVE_ATTACKS))
print("-"*(26 + 13*len(CURVE_ATTACKS)))
for dn, c in curves.items():
    print(f"{dn:26s}" + "".join(f"{q:13.3f}" for q in c[i5]))
print("(IoU at a 5% clean false-positive rate)")

## 9. The figure asked for: raw image, attacked area, detection

One row per image — the untouched picture, where the attack actually went, the detector's
continuous score, and its thresholded mask with the IoU it achieves. The threshold is the
**5%-clean-FP** one from §8, chosen without ever looking at an attacked image.

Two outlines on the attacked panel: **green** is the region drawn for the attack, **cyan** is where
the perturbation actually landed. The gap between them is what v1 was scoring against, and IoU here
is measured against the cyan one.

Set `SHOW` to any key of `ZOO` to see a different attack.

In [ ]:
SHOW  = REF                     # any key of ZOO: "APGD-T", "Square", "low-freq", ...
SCORE = srm_chroma              # the training-free primary detector

with torch.no_grad():
    S   = SCORE(ZOO[SHOW])
    thr = torch.quantile(SCORE(raw).flatten().float()[::7], 1 - 0.05)
    P   = S > thr
    G   = SUPPORT[SHOW]

fig, ax = plt.subplots(N, 4, figsize=(16, 3.9*N)); ax = np.atleast_2d(ax)
for i in range(N):
    region = masks[i,0].cpu().numpy()
    supp   = G[i].float().cpu().numpy()
    ax[i,0].imshow(raw[i].permute(1,2,0).cpu())
    ax[i,0].set_title("raw image" if i == 0 else "")
    ax[i,1].imshow(ZOO[SHOW][i].permute(1,2,0).cpu())
    ax[i,1].contour(region, levels=[.5], colors="lime", linewidths=2)
    ax[i,1].contour(supp,   levels=[.5], colors="cyan", linewidths=1)
    ax[i,1].set_title(f"attacked area ({SHOW})\ngreen = region drawn, cyan = where delta landed"
                      if i == 0 else "")
    ax[i,2].imshow(S[i].cpu(), cmap="inferno")
    ax[i,2].set_title("detector score" if i == 0 else "")
    inter = (P[i] & G[i]).sum().item(); union = (P[i] | G[i]).sum().item()
    ax[i,3].imshow(P[i].cpu(), cmap="gray")
    ax[i,3].contour(supp, levels=[.5], colors="cyan", linewidths=1)
    ax[i,3].set_title(f"detection @5% FP  (IoU {inter/max(union,1):.2f} vs support)")
    for a in ax[i]: a.axis("off")
plt.suptitle(f"Localizing {SHOW} noise. The gap between the green and cyan outlines is the part "
             f"of its own region the attack never used.", y=1.001)
plt.tight_layout(); plt.show()

## 10. What the perturbation does to the actual obstacle detector

The perturbation was crafted against ResNet-50, so its effect on YOLO is pure **transfer** — which
is the realistic threat model, since an attacker poisoning a scraped dataset does not know the
downstream model. Guarded: if `ultralytics` or its weights are unavailable the notebook carries on.

In [ ]:
def to_np(t):
    # ascontiguousarray: permute() leaves a non-contiguous view and ultralytics asserts on it
    # ("Image not contiguous"), which is what killed this cell on the first successful run
    return np.ascontiguousarray((t.permute(1,2,0).cpu().numpy()*255).astype(np.uint8))

try:
    from ultralytics import YOLO
    yolo = YOLO("yolov8n.pt")
    rows = []
    for i in range(N):
        a = yolo.predict(to_np(raw[i]), verbose=False)[0]
        b = yolo.predict(to_np(ZOO[SHOW][i]), verbose=False)[0]
        rows.append((len(a.boxes), len(b.boxes),
                     a.plot()[..., ::-1].copy(), b.plot()[..., ::-1].copy()))

    print(f"{'image':7s} {'boxes clean':>12s} {'boxes attacked':>15s}")
    for i, (na, nb, _, _) in enumerate(rows):
        print(f"{i:<7d} {na:12d} {nb:15d}")
    ta, tb = sum(r[0] for r in rows), sum(r[1] for r in rows)
    print(f"\ntotal detections: {ta} clean -> {tb} attacked "
          f"({100*(tb-ta)/max(ta,1):+.0f}%), while the attacked region covers "
          f"{masks.mean().item()*100:.0f}% of each image")

    fig, ax = plt.subplots(N, 2, figsize=(9, 4.2*N)); ax = np.atleast_2d(ax)
    for i, (na, nb, ia, ib) in enumerate(rows):
        ax[i,0].imshow(ia); ax[i,0].set_title(f"clean - {na} boxes")
        ax[i,1].imshow(ib); ax[i,1].set_title(f"attacked - {nb} boxes")
        for a in ax[i]: a.axis("off")
    plt.suptitle(f"YOLO on clean vs. {SHOW} (crafted on ResNet-50, so this is transfer)", y=1.0)
    plt.tight_layout(); plt.show()
except Exception as e:
    print("YOLO step unavailable:", type(e).__name__, e)

## Takeaway

1. **The ground-truth hypothesis was wrong, and the notebook says so in its own output.** I
   predicted scoring against the drawn region rather than the perturbed pixels cost ~0.13 AUC.
   Measured: support is 99% of the region and SRM scores 0.696 either way. APGD's δ is graded in
   amplitude, not absent — a distinction one RMS number cannot resolve, and I should not have
   asserted it from one. The instrument stays because it is now measured, not assumed.
2. **Cross-channel decorrelation is the feature that generalizes.** Attacks perturb R, G and B
   independently; natural fine detail does not. It is the only feature here that catches Square
   Attack, which is piecewise-constant and therefore invisible to every high-pass residual, and the
   only one that catches smooth L2.
3. **The fitted fusion transfers worse than the features it is built from** -- it beat the best
   single feature on 1 of 6 scored attacks. Fitted on APGD-CE it loses to plain SRM on the L-inf
   attacks and to plain chroma on the rest. Five images is thin, but it is a useful corrective to
   the assumption that learning a combination must help. No blend dominates either, so SRM and
   chroma are reported separately rather than averaged into a single number that is worse at both
   jobs.
4. **Both wrong predictions came from reasoning about data instead of measuring it** -- a
   synthetic proxy in v1, a single summary statistic in v2. The self-checking output added in v2
   is what caught v2's own error: section 6 prints both ground truths side by side, so the claim
   that they differ was refuted by the cell written to support it. That is the pattern worth
   keeping.
5. **The low-frequency poison is still unsolved** -- nothing here exceeds chance on it. `06`
   predicted it, `07` measured it, and this notebook fails against it too, now with better
   instruments and a clearer idea of why.

### Next

- **The low-frequency gap is now the only attack in the zoo with no detector above chance.** The
  untested route from `07` still stands: a **VAE reconstruction residual**, where encode-decode
  projects onto Stable Diffusion's learned natural-image manifold -- a learned prior over content
  rather than a hand-picked band. Chroma decorrelation suggests a cheaper first try: a
  *low-frequency* chroma statistic, since a smooth poison must still perturb the colour channels
  independently.
- **Scale up before trusting any fitted model.** Five images is too thin to conclude more than
  "it did not transfer here"; only `N_IMAGES` changes.
- **Content-conditional normalization** -- bin pixels by an attack-free content proxy, z-score
  within each bin -- measured as a real gain at low false-positive rates in offline testing
  (SRM IoU@1%FP 0.23 -> 0.52) but a wash inside the fusion. Worth a proper test.
- **Close the loop** back into [`../05-Noise-Gate/`](../05-Noise-Gate/): swap `mean(SRM, chroma)`
  in for the HF gate and re-measure quarantine precision and recall on the SD training path.

### References

- Croce & Hein, *Reliable Evaluation of Adversarial Robustness with an Ensemble of Diverse
  Parameter-free Attacks* (AutoAttack), ICML 2020 —
  [arXiv:2003.01690](https://arxiv.org/abs/2003.01690).
- Kim, *Torchattacks: A PyTorch Repository for Adversarial Attacks* —
  [arXiv:2010.01950](https://arxiv.org/abs/2010.01950).
- Liu et al., *Detecting Adversarial Examples Based on Steganalysis* — SRM residuals as
  adversarial-detection features, [arXiv:1806.09186](https://arxiv.org/abs/1806.09186).
- Lorenz et al., *Detecting AutoAttack Perturbations in the Frequency Domain*, ICML 2021 workshop —
  [arXiv:2111.08785](https://arxiv.org/abs/2111.08785). Detects *whether* an image is attacked from
  its spectrum; this notebook asks the harder question of *where*.
- ViT-ReciproCAM — [arXiv:2310.02588](https://arxiv.org/abs/2310.02588), the forward-only,
  batchable scoring idea the detector inherits (see [`../Resources/`](../Resources/)).